In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Leer tabla oro de la compañia padre
df_parent = spark.table(f"{catalog}.{gold_schema}.parent_dim_{data_source}")

# Crear tabla estadarizada "dim_products"
df_parent.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.2_dim_{data_source}")

In [0]:
# Leer tabla oro de la compañia hija
df_child = spark.sql(f"SELECT product_code, division, category, product, variant FROM atlone_enterprise.gold.child_dim_products;")

delta_table = DeltaTable.forName(spark, "atlone_enterprise.gold.2_dim_products")

# Merge con tabla estadarizada "dim_products"
delta_table.alias("target").merge(
    source=df_child.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()